# Tutorial Check

***

This short notebook verifies that your container is working correctly **before** you run the full tutorials. Run each cell from top to bottom. Each section ends with a clear ✅ message if the check passed; if any cell errors out, the message just before it tells you what was being tested.

If everything passes, you're ready to start with `0.qg_tutorial_start.ipynb`.

## 1. Set environment variables, ensure directories and notebooks exist

The tutorials rely on two environment variables:
- `JEDI_BIN` — directory containing the compiled JEDI executables
- `JEDI_EDU` — directory containing the tutorial data, yamls, and helper scripts

In [ ]:
export JEDI_BIN=/home/nonroot/build/bin
export JEDI_EDU=/home/nonroot/shared/EDU
test -d "$JEDI_BIN" && echo "  found $JEDI_BIN"
test -d "$JEDI_EDU" && echo "  found $JEDI_EDU"
echo "✅ environment variables set"

ls /home/nonroot/shared/*ipynb 
if [ $(ls /home/nonroot/shared/*ipynb | wc -l) -ge 10 ]; then
    echo "✅ All noteboooks found"
fi



## 2. Verify the key JEDI executables are present and executable

The tutorials use a handful of `qg_*.x` programs. We list them and check the main one is a real executable.

In [ ]:
set -e
echo "qg executables in $JEDI_BIN:"
ls -1 $JEDI_BIN/qg_*.x | sed 's/^/  /'

# Check one of the key binaries is executable and is an ELF binary
test -x "$JEDI_BIN/qg_forecast.x"
file "$JEDI_BIN/qg_forecast.x" | grep -q -E "ELF|executable" && \
  echo "✅ qg_forecast.x is an executable binary"


In [ ]:
set -e
which python3
python3 --version
echo "✅ python3 is on the PATH"


## 3. Verify Python and required packages

The plotting and analysis scripts in `$JEDI_EDU/plots_scripts` rely on a few standard scientific-Python packages. We try to import each one and report its version.

In [ ]:
set -e
python3 --version
echo "✅ python3 is on the PATH"
python3 - <<'PY'
import importlib, sys
required = ["numpy", "matplotlib", "netCDF4", "yaml"]
missing = []
for pkg in required:
    try:
        m = importlib.import_module(pkg)
        ver = getattr(m, "__version__", "n/a")
        print(f"  {pkg:12s} {ver}")
    except ImportError as e:
        missing.append(pkg)
        print(f"  {pkg:12s} MISSING ({e})")
if missing:
    sys.exit(f"Missing packages: {missing}")
print("✅ all required python packages import OK")
PY


## 4. Run a small Python script

A quick end-to-end test that Python, NumPy, and the file system all work together: generate some data, write it to a NetCDF file, read it back, verify it works.

In [ ]:
set -e
python3 - <<'PY'
import numpy as np
import netCDF4 as nc
import tempfile, os

with tempfile.TemporaryDirectory() as d:
    path = os.path.join(d, "test.nc")
    # write
    with nc.Dataset(path, "w") as ds:
        ds.createDimension("x", 4)
        v = ds.createVariable("data", "f8", ("x",))
        v[:] = np.arange(4, dtype="f8")
    # read
    with nc.Dataset(path, "r") as ds:
        arr = ds.variables["data"][:]
    assert np.allclose(arr, [0, 1, 2, 3]), f"unexpected values: {arr}"
    print(f"  wrote and read back NetCDF: {arr.tolist()}")
print("✅ python netcdf round-trip works")
PY


***

If every cell above ended with a ✅ (or you saw the expected output), your container is set up correctly and you're ready to start with [`0.qg_tutorial_start.ipynb`](0.qg_tutorial_start.ipynb).